# Data Exploration

## Purpose
This notebook performs structured exploratory data analysis (EDA) on the raw dataset.
The goal is to understand data structure, missingness, distributions, and risks
before feature engineering and model training.

## Imports & Configuration

In [ ]:
# Standard libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## Load Project Secrets (Colab / userdata)

In [ ]:
from google.colab import userdata

PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GITHUB_USER = userdata.get("GITHUB_USER")
GCS_BUCKET = userdata.get("GCS_BUCKET")
TRAINING_PREFIX = userdata.get("TRAINING_PREFIX")
REPO_NAME = userdata.get("REPO_NAME")
REGION = userdata.get("REGION")

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
CLONE_PATH = f"/content/{REPO_NAME}"

## Authenticate & Configure gcloud

In [ ]:
!gcloud auth login --quiet
!gcloud config set project $PROJECT_ID
!gcloud config set compute/region $REGION

print("Project:")
!gcloud config get-value project
print("\nUser:")
!gcloud config get-value account
print("\nRegion:")
!gcloud config get-value compute/region

## Clone Repository

In [ ]:
%cd /content
!rm -rf {REPO_NAME}
!git clone {REPO_URL}

# Move to repo root
%cd {CLONE_PATH}

# Verify structure
!ls -lh data/raw

## Load Titanic CSV

In [ ]:
# Path to Titanic CSV relative to repo root
DATA_PATH = "data/raw/train.csv"

# Load CSV
df = pd.read_csv(DATA_PATH)

# Quick preview
df.head()

## Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
df.describe(include="all")

## Missing Value Analysis

### Missing Data Observations
- Identify columns with high missingness (>20%).
- Decide on imputation, drop, or retention.

In [ ]:
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

missing_df

## Target Variable Analysis

### Target Distribution
- Check class balance for survival prediction.



In [ ]:
TARGET = "Survived"
df[TARGET].value_counts()

## Numeric Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

## Categorical Feature Distributions

In [ ]:
categorical_cols = df.select_dtypes(exclude=np.number).columns
for col in categorical_cols:
    print(f"\nValue counts for {col}")
    print(df[col].value_counts().head(10))

## Key Observations

- Target distribution: check for class imbalance.
- Missing data: Age, Cabin require handling.
- Feature engineering implications:
    - Encode Sex as numeric
    - One-hot encode Embarked
    - Create FamilySize = SibSp + Parch + 1
- Drop irrelevant columns: PassengerId, Name, Ticket, Cabin
- Potential leakage risks: none obvious in demo dataset

## Structural Observations

## Key Observations

- **Target distribution:** Survived — Male=577, Female=314.  
- **Notable skew:** Fare is right-skewed; SibSp and Parch have heavy 0 counts.  
- **Missingness strategy:**  
  - Drop `Cabin` entirely (too many missing values).  
  - Drop `Embarked` entirely — only 2 missing rows (~0.2% of data) and column adds minimal predictive signal compared to Sex, Age, Fare, and FamilySize. Dropping simplifies preprocessing with negligible impact.  
  - Impute `Age` missing values with column mean (or median if preferred).  
- **Potential leakage risks:** None obvious in this dataset; avoid including PassengerId, Name, or Ticket as features.  
- **Feature engineering implications:**  
  - Encode `Sex` as numeric (Male=0, Female=1).  
  - Create `FamilySize = SibSp + Parch + 1`.  
  - Optionally bin `Fare` into categories to reduce skew.